## 1. Instalación de dependencias

In [1]:
#%pip install requests beautifulsoup4 pandas

## 2. Imports y funciones auxiliares

In [2]:
# ============================================================
# ARGENPROP SCRAPPER - Departamentos en Venta Capital Federal
# ============================================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os

# ──────────────────────────────────────────────
# UTILIDADES
# ──────────────────────────────────────────────

def clean_text(text):
    if not text: return None
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip() or None


def extract_smart_features(row):
    texto = (str(row.get('Descripción', '')) + " " + str(row.get('Detalles', ''))).lower()
    return pd.Series({
        "Amenities":         1 if any(x in texto for x in ["amenities", "piscina", "pileta", "sum", "parrilla", "gym", "sauna", "laundry"]) else 0,
        "Losa_Central":      1 if any(x in texto for x in ["losa radiante", "calefacción central", "caldera central", "piso radiante"]) else 0,
        "Aire_Acond":        1 if any(x in texto for x in ["aire acondicionado", "split", " a/c", "frío-calor"]) else 0,
        "Apto_Credito":      1 if "apto crédito" in texto or "apto credito" in texto else 0,
        "Cochera":           1 if any(x in texto for x in ["cochera", "espacio guarda coche", "estacionamiento", "guarda coche"]) else 0,
        "Seguridad":         1 if any(x in texto for x in ["vigilancia", "seguridad 24", "tótem", "totem", "encargado", "portero", "guardia", "cámara", "monitoreo"]) else 0,
        "Luminoso":          1 if any(x in texto for x in ["luminoso", "todo luz", "vista abierta", "vista panorámica", "muy soleado", "soleado"]) else 0,
        "Balcon_Aterrazado": 1 if "aterrazado" in texto or "balcón terraza" in texto else 0,
    })


MESES = r'(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre)'


def parse_address(address_raw):
    calle = altura = piso = None
    if not address_raw:
        return calle, altura, piso
    try:
        # En Argenprop el piso viene como "ARAOZ 1200, Piso 8" — separarlo antes de parsear
        piso_match = re.search(r',\s*[Pp]iso\s+(\w+)', address_raw)
        if piso_match:
            piso = piso_match.group(1).strip()
            address_raw = address_raw[:piso_match.start()].strip()  # quitar ", Piso 8" del string

        cleaned = re.sub(r'\bal\s+', '', address_raw, flags=re.IGNORECASE)

        # 2. Si tiene guión, quedarse con fragmento que tenga altura válida (>= 100)
        if '-' in cleaned:
            fragmentos = [f.strip() for f in cleaned.split('-')]
            candidato = None
            for frag in fragmentos:
                if re.search(r'\b\d{3,5}\b', frag):
                    candidato = frag
                    break
            if candidato:
                cleaned = candidato
            else:
                return None, None, piso  # ← conservar piso si ya lo extrajimos

        # 3. Intentar capturar piso inline: "Junín 1615 piso 13" o "Junín 1615 PB"
        if piso is None:
            match_piso = re.search(
                r'([A-Za-záéíóúÁÉÍÓÚñÑü\s\.]+?)\s+(\d{2,5})\s+(piso\s+\d+|PB|pb|p\.b\.)',
                cleaned, re.IGNORECASE
            )
            if match_piso:
                num = int(match_piso.group(2))
                if not (1800 <= num <= 1950 and re.search(MESES, cleaned[:match_piso.start(2)], re.IGNORECASE)):
                    calle    = match_piso.group(1).strip().strip('-').strip('.')
                    altura   = match_piso.group(2).strip()
                    piso_raw = match_piso.group(3).strip()
                    piso     = re.search(r'\d+', piso_raw).group(0) if re.search(r'\d+', piso_raw) else "PB"
                    return calle, altura, piso

        # 4. Iterar números y encontrar altura real
        for m in re.finditer(r'(\d{2,5})(?:[.,\s\-]|$)', cleaned + ' '):
            num = int(m.group(1))
            if num < 100:
                continue
            contexto_previo = cleaned[:m.start()].lower()
            es_año_historico = (1800 <= num <= 1950) and bool(re.search(MESES, contexto_previo, re.IGNORECASE))
            if es_año_historico:
                continue
            calle_raw = cleaned[:m.start()].strip().strip('-').strip('.').strip()
            if not calle_raw:
                continue
            calle  = calle_raw
            altura = str(num)
            return calle, altura, piso

    except:
        pass
    return None, None, piso


def parse_card_argenprop(item):
    # Link
    link_tag = item.find('a', class_='card')
    if not link_tag:
        return None
    link = "https://www.argenprop.com" + link_tag['href']

    # Precio y expensas
    price_block = item.find('p', class_='card__price')
    price_text  = clean_text(price_block.text) if price_block else ""
    p_match  = re.search(r'(USD|ARS|\$)\s?([\d.]+)', price_text or "")
    precio   = p_match.group(0) if p_match else "Consultar"
    e_match  = re.search(r'\+\s?\$?\s?([\d.]+)', price_text or "")
    expensas = e_match.group(0) if e_match else None

    # Dirección — viene como "ARAOZ 1200, Piso 8"
    addr_tag    = item.find('p', class_='card__address')
    raw_address = clean_text(addr_tag.text) if addr_tag else None
    calle, altura, piso = parse_address(raw_address)

    # Barrio — en Argenprop está en card__title--primary: "Depto en Venta en Palermo, Capital Federal"
    barrio_tag = item.find('p', class_='card__title--primary')
    barrio = None
    if barrio_tag:
        barrio_text = clean_text(barrio_tag.text)
        # Extraer solo el barrio: "Departamento en Venta en Palermo, Capital Federal" → "Palermo"
        b_match = re.search(r'\ben\s+([^,]+),', barrio_text, re.IGNORECASE)
        barrio = b_match.group(1).strip() if b_match else barrio_text

    # Características
    feat_tag = item.find('ul', class_='card__main-features')
    features = clean_text(feat_tag.get_text(separator=' ')) if feat_tag else None

    # Descripción corta
    desc_tag = item.find('p', class_='card__description') or item.find('p', class_='card__info')
    desc     = clean_text(desc_tag.text) if desc_tag else None

    return link, precio, expensas, calle, altura, piso, barrio, features, desc

print("✅ Funciones cargadas correctamente.")


✅ Funciones cargadas correctamente.


## 3. Función principal del scrapper

In [3]:
def run_scrapper_argenprop(enlace, operacion, max_pages=3):
    BASE_URL   = enlace
    HEADERS    = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}
    OUTPUT_DIR = "output"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_data   = []
    seen_links = set()

    for i in range(1, max_pages + 1):
        url = f"{BASE_URL}?pagina-{i}" if i > 1 else BASE_URL
        print(f"\n--- PÁGINA {i} ---")
        print(f"URL: {url}")

        try:
            r    = requests.get(url, headers=HEADERS, timeout=15)
            soup = BeautifulSoup(r.content, 'html.parser')
            items = soup.find_all('div', class_='listing__item')

            if not items:
                print("⚠️ No se encontraron items. Deteniendo.")
                break

            print(f"  → {len(items)} propiedades encontradas")

            for item in items:
                try:
                    result = parse_card_argenprop(item)
                    if not result:
                        continue
                    link, precio, expensas, calle, altura, piso, barrio, features, desc = result

                    if link in seen_links:
                        continue
                    seen_links.add(link)

                    print(f"  → {calle} {altura} | {precio}")

                    all_data.append({
                        "Precio":      precio,
                        "Expensas":    expensas,
                        "Calle":       calle,
                        "Altura":      altura,
                        "Piso":        piso,
                        "Barrio":      barrio,
                        "Detalles":    features,
                        "Descripción": desc,
                        "Link":        link,
                    })
                except Exception as e:
                    print(f"    ⚠️ Error en card: {e}")
                    continue

            time.sleep(1.5)

        except Exception as e:
            print(f"❌ Error en página {i}: {e}")
            break

    if not all_data:
        print("⚠️ No se obtuvieron datos.")
        return None

    df = pd.DataFrame(all_data)
    df = df.replace("N/A", None)
    features_df = df.apply(extract_smart_features, axis=1)
    df = pd.concat([df, features_df], axis=1)

    filename = f"argenprop_{operacion}_{int(time.time())}.tsv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(filepath, sep='\t', index=False, encoding='utf-8-sig')
    print(f"\n✅ ¡Éxito! Guardado en: {filepath}")
    return df

print("✅ run_scrapper_argenprop() lista.")

✅ run_scrapper_argenprop() lista.


## 4. ▶️ Ejecutar el scrapper

# Alquiler

In [4]:
df_alquiler = run_scrapper_argenprop("https://www.argenprop.com/departamentos/venta/capital-federal", 'alquiler', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.argenprop.com/departamentos/venta/capital-federal
  → 20 propiedades encontradas
  → ARAOZ 1200 | USD 330.000
  → Jerónimo Salguero 2700 | USD 2.500.000
  → Bulnes 1600 | USD 150.000
  → Av. Gral. Indalecio Chenaut 1900 | USD 181.000
  → Honduras 3900 | USD 270.000
  → Castex 3300 | USD 570.000
  → GURRUCHAGA 2100 | USD 98.000
  → BORGES JORGE LUIS 2100 | USD 240.000
  → Soler 5800 | USD 129.000
  → Jorge Luis Borges 2400 | USD 219.000
  → Gurruchaga 1100 | USD 98.000
  → AV FRANCISCO ACUÑA DE FIGUEROA 1200 | USD 113.076
  → RIVADAVIA, AV 5200 | USD 68.000
  → CONESA 2500 | USD 105.000
  → AMENABAR 2100 | USD 129.000
  → Balbin 2400 | USD 620.000
  → Jerónimo Salguero 1900 | USD 198.000
  → Santa Rosa 5000 | USD 89.000
  → Bulnes 1300 | USD 288.000
  → Fray Justo Santa María De Oro 2200 | USD 189.000

--- PÁGINA 2 ---
URL: https://www.argenprop.com/departamentos/venta/capital-federal?pagina-2
  → 20 propiedades encontradas
  → ZAPATA 300 | USD 91.000


In [5]:
if df_alquiler is not None:
    print(f"Total propiedades: {len(df_alquiler)}")
    display(df_alquiler.head(10))

    cols = ["Amenities","Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_alquiler[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 60


,Precio,Expensas,Calle,Altura,Piso,Barrio,Detalles,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,USD 330.000,+ $203.300,ARAOZ,1200,8,Venta en Palermo,90 m² cubie. 3 dorm. A Estrenar,Venta 4 ambientes con balcón palermo + cochera...,https://www.argenprop.com/departamento-en-vent...,1,0,0,0,1,1,1,0
1,USD 2.500.000,+ $2.400.000,Jerónimo Salguero,2700,None,Venta en Palermo Chico,275 m² cubie. 3 dorm. 17 años,“Torre bellini” de revista! Impecable piso muy...,https://www.argenprop.com/departamento-en-vent...,1,0,1,1,1,1,1,0
2,USD 150.000,+ $260.000,Bulnes,1600,None,Venta en Botanico,60 m² cubie. 2 dorm. 30 años,Impecable! Balcón corrido al frente con vista ...,https://www.argenprop.com/departamento-en-vent...,0,0,1,1,1,1,1,0
3,USD 181.000,+ $335.000,Av. Gral. Indalecio Chenaut,1900,None,Venta en Las Cañitas,75 m² cubie. 1 dorm. 20 años,Impecable 2 ambientes en dúplex! Balcón al fre...,https://www.argenprop.com/departamento-en-vent...,1,0,1,0,0,1,1,0
4,USD 270.000,+ $300.000,Honduras,3900,2º,Venta en Palermo Viejo,87 m² cubie. 2 dorm. 20 años,"Venta semipiso 4 ambientes, dos baños completo...",https://www.argenprop.com/departamento-en-vent...,1,0,1,0,1,0,1,1
5,USD 570.000,+ $1.000.000,Castex,3300,None,Venta en Palermo Chico,140 m² cubie. 3 dorm. 40 años,Palermo chico espectacular piso alto con vista...,https://www.argenprop.com/departamento-en-vent...,0,1,0,0,1,1,1,1
6,USD 98.000,+ $150.000,GURRUCHAGA,2100,5,Venta en Palermo,31 m² cubie. 15 años 1 baño,Venta - departamento 1 ambiente al frente en p...,https://www.argenprop.com/departamento-en-vent...,0,1,0,0,0,0,0,0
7,USD 240.000,+ $340.000,BORGES JORGE LUIS,2100,6,Venta en Palermo Soho,67 m² cubie. 2 dorm. 5 años,Departamento en venta 3 ambientes en palermo s...,https://www.argenprop.com/departamento-en-vent...,1,1,1,0,0,0,1,0
8,USD 129.000,+ $240.000,Soler,5800,1,Venta en Palermo Hollywood,41 m² cubie. 1 dorm. 10 años,"Unidad de 2 ambientes, muy luminoso y en una e...",https://www.argenprop.com/departamento-en-vent...,0,0,0,0,0,0,1,0
9,USD 219.000,+ $200.000,Jorge Luis Borges,2400,None,Venta en Botanico,75 m² cubie. 3 dorm. A Estrenar,Departamento dúplex de 4 ambientes reciclado a...,https://www.argenprop.com/departamento-en-vent...,0,0,1,0,0,0,1,0



📊 % con cada característica:
Amenities            45.0%
Losa_Central         11.7%
Aire_Acond           45.0%
Apto_Credito         13.3%
Cochera              33.3%
Seguridad            33.3%
Luminoso             63.3%
Balcon_Aterrazado    10.0%
dtype: object


# Alquiler Temporal

In [6]:
df_alquiler_temporal = run_scrapper_argenprop('https://www.argenprop.com/departamentos/alquiler-temporal/capital-federal', 'alquiler_temporal', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.argenprop.com/departamentos/alquiler-temporal/capital-federal
  → 20 propiedades encontradas
  → JUAN MARÍA GUTIÉRREZ 3900 | USD 3.000
  → Soldado De La Independencia 1100 | $ 790.000
  → Manuel Nicolas Savio 400 | USD 950
  → Av Gral Luis Maria Campos 1100 | $ 1.300.000
  → Concepción Arenal 3500 | $ 650.000
  → Costa Rica 4300 | USD 1.400
  → CORONEL DIAZ 1500 | USD 1.100
  → Gral Lucio Mansilla 2600 | USD 900
  → Paraguay 1200 | USD 1.000
  → Ayacucho 2100 | USD 1.500
  → Presidente José Evaristo Uriburu 1600 | Consultar
  → Santa fe 2600 | USD 700
  → Guatemala 5653 | USD 1.500
  → None None | USD 880
  → Charcas 4300 | USD 750
  → Rodriguez Peña 1400 | USD 1.500
  → 3 DE FEBRERO 2500 | USD 800
  → Delfin Huergo 300 | USD 1.150
  → Concepción Arenal 2300 | USD 1.000
  → ECUADOR 1500 | USD 1.100

--- PÁGINA 2 ---
URL: https://www.argenprop.com/departamentos/alquiler-temporal/capital-federal?pagina-2
  → 20 propiedades encontradas
  → PACHECO DE MEL

In [7]:
if df_alquiler_temporal is not None:
    print(f"Total propiedades: {len(df_alquiler_temporal)}")
    display(df_alquiler_temporal.head(10))

    cols = ["Amenities","Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_alquiler_temporal[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 60


,Precio,Expensas,Calle,Altura,Piso,Barrio,Detalles,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,USD 3.000,+ $640.000,JUAN MARÍA GUTIÉRREZ,3900,None,Alquiler Temporal en Palermo Chico,130 m² cubie. 2 dorm. 15 años,Torre de categoría! Impecable semipiso de revi...,https://www.argenprop.com/departamento-en-alqu...,1,0,1,0,1,1,1,1
1,$ 790.000,+ $234.000,Soldado De La Independencia,1100,6,Alquiler Temporal en Las Cañitas,32 m² cubie. 1 baño Monoam.,Reservado. Divino monoambiente dueño directo. ...,https://www.argenprop.com/departamento-en-alqu...,1,0,0,0,0,0,1,0
2,USD 950,None,Manuel Nicolas Savio,400,4,Alquiler Temporal en Las Cañitas,65 m² cubie. 2 dorm. 2 baños,Excelente unidad. Amplio y luminoso departamen...,https://www.argenprop.com/departamento-en-alqu...,0,0,1,0,0,0,1,0
3,$ 1.300.000,+ $325.000,Av Gral Luis Maria Campos,1100,2,Alquiler Temporal en Las Cañitas,75 m² cubie. 2 dorm. 45 años,Alquiler: $ 1.300.000 ó u$s 950. Expensas: $ 3...,https://www.argenprop.com/departamento-en-alqu...,0,0,1,0,0,1,0,0
4,$ 650.000,+ $199.600,Concepción Arenal,3500,None,Alquiler Temporal en Palermo Hollywood,31 m² cubie. 5 años 1 baño,Alquiler monoambiente amoblado en palermo con ...,https://www.argenprop.com/departamento-en-alqu...,1,1,1,0,0,0,0,0
5,USD 1.400,None,Costa Rica,4300,None,Alquiler Temporal en Palermo,48 m² cubie. 1 dorm. 5 años,Excelente unidad de 2 ambientes ubicada en cos...,https://www.argenprop.com/departamento-en-alqu...,1,0,1,0,0,0,0,0
6,USD 1.100,None,CORONEL DIAZ,1500,None,Alquiler Temporal en Palermo,75 m² cubie. 2 dorm. 1 baño,Alquiler temporario departamento amueblado de ...,https://www.argenprop.com/departamento-en-alqu...,0,0,0,0,0,0,1,0
7,USD 900,+ $300.000,Gral Lucio Mansilla,2600,None,Alquiler Temporal en Recoleta,70 m² cubie. 1 dorm. 12 años,Impecable 2 ambientes! Espectacular balcón ter...,https://www.argenprop.com/departamento-en-alqu...,1,0,1,0,0,1,1,1
8,USD 1.000,+ $325.000,Paraguay,1200,None,Alquiler Temporal en Recoleta,100 m² cubie. 2 dorm. 30 años,Impecable piso alto! Balcón francés al frente ...,https://www.argenprop.com/departamento-en-alqu...,0,0,1,0,0,0,1,0
9,USD 1.500,+ $620.000,Ayacucho,2100,None,Alquiler Temporal en Recoleta,95 m² cubie. 2 dorm. 40 años,Impecable semipiso! Balcón corrido con vista a...,https://www.argenprop.com/departamento-en-alqu...,0,1,1,0,0,1,1,0



📊 % con cada característica:
Amenities            43.3%
Losa_Central          6.7%
Aire_Acond           46.7%
Apto_Credito          0.0%
Cochera              11.7%
Seguridad            18.3%
Luminoso             48.3%
Balcon_Aterrazado     5.0%
dtype: object


# Venta

In [8]:
df_venta = run_scrapper_argenprop('https://www.argenprop.com/departamentos/venta/capital-federal', 'venta', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.argenprop.com/departamentos/venta/capital-federal
  → 20 propiedades encontradas
  → ARAOZ 1200 | USD 330.000
  → Jerónimo Salguero 2700 | USD 2.500.000
  → Bulnes 1600 | USD 150.000
  → Av. Gral. Indalecio Chenaut 1900 | USD 181.000
  → Honduras 3900 | USD 270.000
  → Castex 3300 | USD 570.000
  → GURRUCHAGA 2100 | USD 98.000
  → BORGES JORGE LUIS 2100 | USD 240.000
  → Soler 5800 | USD 129.000
  → Jorge Luis Borges 2400 | USD 219.000
  → Gurruchaga 1100 | USD 98.000
  → AV FRANCISCO ACUÑA DE FIGUEROA 1200 | USD 113.076
  → RIVADAVIA, AV 5200 | USD 68.000
  → CONESA 2500 | USD 105.000
  → AMENABAR 2100 | USD 129.000
  → Balbin 2400 | USD 620.000
  → Jerónimo Salguero 1900 | USD 198.000
  → Santa Rosa 5000 | USD 89.000
  → Bulnes 1300 | USD 288.000
  → Fray Justo Santa María De Oro 2200 | USD 189.000

--- PÁGINA 2 ---
URL: https://www.argenprop.com/departamentos/venta/capital-federal?pagina-2
  → 20 propiedades encontradas
  → ZAPATA 300 | USD 91.000


In [9]:
if df_venta is not None:
    print(f"Total propiedades: {len(df_venta)}")
    display(df_venta.head(10))

    cols = ["Amenities","Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_venta[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 60


,Precio,Expensas,Calle,Altura,Piso,Barrio,Detalles,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,USD 330.000,+ $203.300,ARAOZ,1200,8,Venta en Palermo,90 m² cubie. 3 dorm. A Estrenar,Venta 4 ambientes con balcón palermo + cochera...,https://www.argenprop.com/departamento-en-vent...,1,0,0,0,1,1,1,0
1,USD 2.500.000,+ $2.400.000,Jerónimo Salguero,2700,None,Venta en Palermo Chico,275 m² cubie. 3 dorm. 17 años,“Torre bellini” de revista! Impecable piso muy...,https://www.argenprop.com/departamento-en-vent...,1,0,1,1,1,1,1,0
2,USD 150.000,+ $260.000,Bulnes,1600,None,Venta en Botanico,60 m² cubie. 2 dorm. 30 años,Impecable! Balcón corrido al frente con vista ...,https://www.argenprop.com/departamento-en-vent...,0,0,1,1,1,1,1,0
3,USD 181.000,+ $335.000,Av. Gral. Indalecio Chenaut,1900,None,Venta en Las Cañitas,75 m² cubie. 1 dorm. 20 años,Impecable 2 ambientes en dúplex! Balcón al fre...,https://www.argenprop.com/departamento-en-vent...,1,0,1,0,0,1,1,0
4,USD 270.000,+ $300.000,Honduras,3900,2º,Venta en Palermo Viejo,87 m² cubie. 2 dorm. 20 años,"Venta semipiso 4 ambientes, dos baños completo...",https://www.argenprop.com/departamento-en-vent...,1,0,1,0,1,0,1,1
5,USD 570.000,+ $1.000.000,Castex,3300,None,Venta en Palermo Chico,140 m² cubie. 3 dorm. 40 años,Palermo chico espectacular piso alto con vista...,https://www.argenprop.com/departamento-en-vent...,0,1,0,0,1,1,1,1
6,USD 98.000,+ $150.000,GURRUCHAGA,2100,5,Venta en Palermo,31 m² cubie. 15 años 1 baño,Venta - departamento 1 ambiente al frente en p...,https://www.argenprop.com/departamento-en-vent...,0,1,0,0,0,0,0,0
7,USD 240.000,+ $340.000,BORGES JORGE LUIS,2100,6,Venta en Palermo Soho,67 m² cubie. 2 dorm. 5 años,Departamento en venta 3 ambientes en palermo s...,https://www.argenprop.com/departamento-en-vent...,1,1,1,0,0,0,1,0
8,USD 129.000,+ $240.000,Soler,5800,1,Venta en Palermo Hollywood,41 m² cubie. 1 dorm. 10 años,"Unidad de 2 ambientes, muy luminoso y en una e...",https://www.argenprop.com/departamento-en-vent...,0,0,0,0,0,0,1,0
9,USD 219.000,+ $200.000,Jorge Luis Borges,2400,None,Venta en Botanico,75 m² cubie. 3 dorm. A Estrenar,Departamento dúplex de 4 ambientes reciclado a...,https://www.argenprop.com/departamento-en-vent...,0,0,1,0,0,0,1,0



📊 % con cada característica:
Amenities            45.0%
Losa_Central         11.7%
Aire_Acond           45.0%
Apto_Credito         13.3%
Cochera              33.3%
Seguridad            33.3%
Luminoso             63.3%
Balcon_Aterrazado    10.0%
dtype: object
